# fintrack — 04 API Response Validation
**Purpose**: Cross-check API responses against direct PostgreSQL queries.  
**Version**: 1.0 — 2026-03-24  

Each section runs the same query two ways — via the API and directly against  
the DB — then compares the results. Any discrepancy is flagged as FAIL.

In [39]:
import os
import httpx
import psycopg2
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

API_BASE = f"http://{os.getenv('API_HOST','192.168.1.170')}:{os.getenv('API_PORT','8000')}"
EMAIL    = os.getenv('FINTRACK_EMAIL',    'your@email.com')
PASSWORD = os.getenv('FINTRACK_PASSWORD', 'yourpassword')
DB_CONN  = psycopg2.connect(
    host=os.getenv('DB_HOST','192.168.1.169'),
    port=int(os.getenv('DB_PORT','5432')),
    dbname=os.getenv('DB_NAME','fintrack'),
    user=os.getenv('DB_USER','fintrack'),
    password=os.getenv('DB_PASSWORD','')
)

r = httpx.post(f'{API_BASE}/api/v1/auth/login',
               json={'email': EMAIL, 'password': PASSWORD}, timeout=10)
TOKEN   = r.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
USER_ID = r.json()['user_id']

def db_query(sql, params=None):
    cur = DB_CONN.cursor()
    cur.execute(sql, params or ())
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def check(label, api_val, db_val, tolerance=0.02):
    diff = abs(float(api_val) - float(db_val))
    status = 'PASS' if diff <= tolerance else 'FAIL'
    icon   = '✓' if status == 'PASS' else '✗'
    print(f'  {icon} {label}')
    print(f'     API: {api_val:>12,.2f}   DB: {db_val:>12,.2f}   diff: {diff:.2f}')
    return status == 'PASS'

print('Setup complete')

Setup complete


## Check 1 — Grand total

In [40]:
# API
api_cat  = httpx.get(f'{API_BASE}/api/v1/analytics/category-summary',
                     headers=HEADERS, params={'year': 2025}, timeout=15).json()
api_total = api_cat['grand_total']

# DB
db_total = db_query("""
    SELECT SUM(amount) AS total
    FROM transactions
    WHERE user_id = %s AND year_num = 2025
""", (USER_ID,))['total'][0]

print('Check 1 — Grand Total (2025)')
check('Grand total', api_total, db_total)

Check 1 — Grand Total (2025)
  ✓ Grand total
     API:    53,232.72   DB:    53,232.72   diff: 0.00


True

## Check 2 — Category totals

In [33]:
# API category totals
api_cats = {f"{c['category']}|{c['subcategory']}": c['total']
            for c in api_cat['categories']}

# DB category totals
db_cats_df = db_query("""
    SELECT category_name,
           COALESCE(subcategory, category_name) AS subcategory,
           ROUND(SUM(amount)::NUMERIC, 2) AS total
    FROM transactions
    WHERE user_id = %s AND year_num = 2025
    GROUP BY category_name, subcategory
    ORDER BY total DESC
""", (USER_ID,))

print('Check 2 — Category totals (2025)')
all_pass = True
for _, row in db_cats_df.iterrows():
    key     = f"{row['category_name']}|{row['subcategory']}"
    api_val = api_cats.get(key, 0)
    db_val  = float(row['total'])
    ok = check(key[:50], api_val, db_val)
    if not ok:
        all_pass = False

print()
print('OVERALL:', 'ALL PASS ✓' if all_pass else 'FAILURES FOUND ✗')

Check 2 — Category totals (2025)
  ✓ Air Travel|Air Travel
     API:    12,727.67   DB:    12,727.67   diff: 0.00
  ✓ Home Improvement|Home Improvement
     API:     8,296.46   DB:     8,296.46   diff: 0.00
  ✓ Other|Other
     API:     4,586.37   DB:     4,586.37   diff: 0.00
  ✓ Shopping|General Retail
     API:     3,355.59   DB:     3,355.59   diff: 0.00
  ✓ Groceries|Ethnic Grocery
     API:     3,089.16   DB:     3,089.16   diff: 0.00
  ✓ Groceries|Warehouse Club - Food
     API:     2,759.42   DB:     2,759.42   diff: 0.00
  ✓ Groceries|Grocery Store
     API:     2,424.12   DB:     2,424.12   diff: 0.00
  ✓ Health|Medical & Dental
     API:     1,650.88   DB:     1,650.88   diff: 0.00
  ✓ Transport|Fuel
     API:     1,573.11   DB:     1,573.11   diff: 0.00
  ✓ Hotel|Hotel
     API:     1,496.23   DB:     1,496.23   diff: 0.00
  ✓ Car Rental|Car Rental
     API:     1,392.04   DB:     1,392.04   diff: 0.00
  ✓ Dining|Restaurant
     API:     1,225.23   DB:     1,225.23   diff: 

## Check 3 — Monthly trend totals

In [34]:
# Re-authenticate before Check 3
r_auth = httpx.post(f'{API_BASE}/api/v1/auth/login',
                    json={'email': 'mahajani6630@gmail.com', 'password': 'Akamo@6630'}, timeout=10)
TOKEN   = r_auth.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
print('Token refreshed')

# Now fetch trend
api_trend = httpx.get(f'{API_BASE}/api/v1/analytics/trend',
                      headers=HEADERS, params={'year': 2025}, timeout=15).json()
print('Keys in response:', list(api_trend.keys()))

Token refreshed


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [35]:
api_trend = httpx.get(f'{API_BASE}/api/v1/analytics/trend',
                      headers=HEADERS, params={'year': 2025}, timeout=15).json()

print('Category filter active:', api_trend.get('category'))
print('Total from API:', api_trend.get('total'))
print('Number of months:', len(api_trend.get('months', [])))
print()
for m in api_trend.get('months', []):
    print(f"  {m['month']:12s}  ${m['total']:>10,.2f}  txns: {m['txn_count']}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [42]:
# API trend
#print('category filter:', trend_data.get('category'))
print('months count:', len(api_months))
print('first month total:', list(api_months.values())[0])

api_trend = httpx.get(f'{API_BASE}/api/v1/analytics/trend',
                      headers=HEADERS, params={'year': 2025}, timeout=15).json()
api_months = {m['month']: m['total'] for m in api_trend['months']}

# DB trend
db_trend = db_query("""
    SELECT TO_CHAR(DATE_TRUNC('month', txn_date), 'Mon YYYY') AS month_label,
           ROUND(SUM(amount)::NUMERIC, 2) AS total
    FROM transactions
    WHERE user_id = %s AND year_num = 2025
    GROUP BY month_label, EXTRACT(MONTH FROM txn_date), EXTRACT(YEAR FROM txn_date)
    ORDER BY EXTRACT(YEAR FROM txn_date), EXTRACT(MONTH FROM txn_date)
""", (USER_ID,))

print('Check 3 — Monthly totals (2025)')
all_pass = True
for _, row in db_trend.iterrows():
    api_val = api_months.get(row['month_label'], 0)
    ok = check(row['month_label'], api_val, float(row['total']))
    if not ok:
        all_pass = False
print()
print('OVERALL:', 'ALL PASS ✓' if all_pass else 'FAILURES FOUND ✗')

months count: 12
first month total: 129.0


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Check 4 — Essential vs non-essential split

In [12]:
# API
api_essential    = api_cat['essential_total']
api_nonessential = api_cat['nonessential_total']

# DB — join to categories to get is_essential flag
db_split = db_query("""
    SELECT
        COALESCE(c.is_essential, false) AS is_essential,
        ROUND(SUM(t.amount)::NUMERIC, 2) AS total
    FROM transactions t
    LEFT JOIN categories c
        ON c.user_id = t.user_id
        AND c.name = t.category_name
        AND (c.subcategory = t.subcategory OR
             (c.subcategory IS NULL AND t.subcategory IS NULL))
    WHERE t.user_id = %s AND t.year_num = 2025
    GROUP BY c.is_essential
""", (USER_ID,))

db_essential    = float(db_split[db_split['is_essential']==True]['total'].sum())
db_nonessential = float(db_split[db_split['is_essential']==False]['total'].sum())

print('Check 4 — Essential split (2025)')
check('Essential total',    api_essential,    db_essential)
check('Non-essential total', api_nonessential, db_nonessential)

Check 4 — Essential split (2025)
  ✓ Essential total
     API:    23,608.49   DB:    23,608.49   diff: 0.00
  ✓ Non-essential total
     API:    29,624.23   DB:    29,624.23   diff: 0.00


True

## Check 5 — Transaction count

In [13]:
# API count
r_txn = httpx.get(f'{API_BASE}/api/v1/transactions',
                  headers=HEADERS,
                  params={'password': PASSWORD, 'page_size': 1, 'year': 2025},
                  timeout=15).json()
api_count = r_txn['total']

# DB count
db_count = db_query("""
    SELECT COUNT(*) AS cnt FROM transactions
    WHERE user_id = %s AND year_num = 2025
""", (USER_ID,))['cnt'][0]

print('Check 5 — Transaction count (2025)')
ok = api_count == db_count
print(f'  {"✓" if ok else "✗"} Count   API: {api_count}   DB: {db_count}')

print()
print('=== VALIDATION SUMMARY ===')
print(f'  Grand total:  ${api_cat["grand_total"]:,.2f}')
print(f'  Transactions: {api_count}')
print(f'  Essential:    {api_cat["essential_pct"]}%  |  Non-essential: {api_cat["nonessential_pct"]}%')

Check 5 — Transaction count (2025)
  ✓ Count   API: 655   DB: 655

=== VALIDATION SUMMARY ===
  Grand total:  $53,232.72
  Transactions: 655
  Essential:    44.3%  |  Non-essential: 55.7%
